# Reproducing SAGE on ESConv

**Paper:** Yang, Hong, Jo, Oh. *SAGE: Self-retrieval-augmented generative LLM for emotional support conversation*. Expert Systems with Applications 313 (2026) 131524. [doi:10.1016/j.eswa.2026.131524](https://doi.org/10.1016/j.eswa.2026.131524)

This notebook holds the whole reproduction in one file. It covers the data preparation for both datasets the paper uses (ESConv and ESC-Eval), the paper's metrics, a BM25 floor, the open-LLM baselines, a re-implementation of SAGE and the ESC-Eval role-play evaluation. The authors released no code, so every setting the paper leaves out is marked **ASSUMPTION** in the code. The full list of gaps and inconsistencies is in `AUDIT.md` in the repo.

**How to run.** Use Jupyter or Colab with an NVIDIA GPU: a T4 or better, 16 GB VRAM recommended. Run the cells top to bottom. Leave `QUICK = True` for a ~20-minute check on 200 test examples, then set it to `False` for the full runs (several hours). On a machine without a GPU, only sections 1-4 (data, metrics, BM25) are practical.

## 0. Setup

In [ ]:
# Installs what Colab/Jupyter usually lacks; harmless if already present.
%pip -q install transformers accelerate sentencepiece nltk scikit-learn rank-bm25 peft huggingface_hub
import torch
if torch.cuda.is_available():
    %pip -q install bitsandbytes
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

In [ ]:
import os

# ---- Configuration -----------------------------------------------------------
QUICK = True                     # True: 200 test examples, 1 epoch. False: full paper-scale runs.
SPLIT = "paper"                  # "paper" = 7:1:2 as in SAGE; "zcj" = 70/15/15 split used by earlier baselines
WORKDIR = os.environ.get("SAGE_WORKDIR", "sage_work")   # data, checkpoints and results go here
T5_BASE = os.environ.get("SAGE_T5_BASE", "t5-base")     # SAGE backbone (Appendix B)
LLMS = os.environ.get("SAGE_LLMS", "Qwen/Qwen2.5-7B-Instruct,mistralai/Mistral-7B-Instruct-v0.3").split(",")
# Add "meta-llama/Llama-3.1-8B-Instruct" to LLMS after logging in to Hugging Face with access to that gated model.
LOAD_4BIT = torch.cuda.is_available()  # 4-bit (needs CUDA) fits 7-8B models on a 16 GB T4; set False on GPUs with 24 GB+

# On Colab, keep outputs on Google Drive so a disconnect loses nothing.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/sage_work"
except ImportError:
    pass
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
LIMIT = 200 if QUICK else 0
EPOCHS = 1 if QUICK else 3
print("Working in", os.getcwd(), "| QUICK =", QUICK)

## 1. Data: ESConv and ESC-Eval

Downloads ESConv (Liu et al., 2021; academic research use only) and the ESC-Eval role cards, which section 7 uses. ESConv becomes one example per supporter turn, holding the dialogue context, the gold response and its strategy.

In [ ]:
import json
import os
import random

STRATEGIES = [
    "Question",
    "Restatement or Paraphrasing",
    "Reflection of feelings",
    "Self-disclosure",
    "Affirmation and Reassurance",
    "Providing Suggestions",
    "Information",
    "Others",
]


def norm(text):
    return " ".join(text.strip().split())


def reformat(d):
    # Mirrors codes_zcj process_data, so the zcj split reproduces exactly.
    dial = []
    for u in d["dialog"]:
        turn = {"text": norm(u["content"]), "speaker": "usr" if u["speaker"] == "seeker" else "sys"}
        if turn["speaker"] == "sys":
            turn["strategy"] = u["annotation"]["strategy"]
        dial.append(turn)
    return {
        "emotion_type": d["emotion_type"],
        "problem_type": d["problem_type"],
        "situation": d["situation"],
        "dialog": dial,
    }


def split_dialogs(data, mode):
    data = list(data)
    random.seed(13)
    random.shuffle(data)
    n = len(data)
    if mode == "zcj":
        dev, test = int(0.15 * n), int(0.15 * n)
        return {"valid": data[:dev], "test": data[dev:dev + test], "train": data[dev + test:]}
    train, dev = round(0.7 * n), round(0.1 * n)
    return {"train": data[:train], "valid": data[train:train + dev], "test": data[train + dev:]}


def to_examples(dialogs, split):
    out = []
    for di, d in enumerate(dialogs):
        for ti, turn in enumerate(d["dialog"]):
            # Skip opening supporter turns with no seeker utterance before them.
            if turn["speaker"] != "sys" or not any(t["speaker"] == "usr" for t in d["dialog"][:ti]):
                continue
            ctx = d["dialog"][:ti]
            last_usr = next(t["text"] for t in reversed(ctx) if t["speaker"] == "usr")
            out.append({
                "id": f"{split}-{di}-{ti}",
                "dialog_idx": di,
                "turn_idx": ti,
                "num_prev_turns": ti,
                "situation": d["situation"],
                "emotion_type": d["emotion_type"],
                "problem_type": d["problem_type"],
                "context": [{"speaker": t["speaker"], "text": t["text"], "strategy": t.get("strategy")} for t in ctx],
                "query": last_usr,
                "response": turn["text"],
                "strategy": turn["strategy"],
            })
    return out


In [ ]:
import urllib.request
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/ESConv.json"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/thu-coai/Emotional-Support-Conversation/main/ESConv.json", "data/ESConv.json")
os.makedirs("data/esceval", exist_ok=True)
ESC_RAW = "https://raw.githubusercontent.com/haidequanbu/ESC-Eval/9ad46e7b5e247e824dae4633910eaa82be668beb/data/"
for f in ("card_high_en.json", "card_high_zh.json"):
    if not os.path.exists(f"data/esceval/{f}"):
        urllib.request.urlretrieve(ESC_RAW + f, f"data/esceval/{f}")
if not os.path.exists("data/esceval/escrank_prompts_en.json"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Floxy1991/Sage/main/data/esceval/escrank_prompts_en.json",
                               "data/esceval/escrank_prompts_en.json")
print("ESC-Eval role cards:", len(json.load(open("data/esceval/card_high_en.json"))), "English,",
      len(json.load(open("data/esceval/card_high_zh.json"))), "Chinese")
esconv = [reformat(d) for d in json.load(open("data/ESConv.json"))]
for mode in ("paper", "zcj"):
    os.makedirs(f"data/{mode}", exist_ok=True)
    for split, dialogs in split_dialogs(esconv, mode).items():
        ex = to_examples(dialogs, split)
        with open(f"data/{mode}/{split}.jsonl", "w") as f:
            for e in ex:
                f.write(json.dumps(e) + "\n")
        print(f"{mode:5s} {split:5s} dialogs={len(dialogs):4d} examples={len(ex):5d}")

### Check: strategy distribution

The SAGE paper's Table A.9 says, for example, Self-disclosure is 1.5% and Restatement 0.5%. ESConv's own counts are below. They cover supporter turns only, because only those carry strategy labels.

In [ ]:
from collections import Counter
strat = Counter(t["strategy"] for d in esconv for t in d["dialog"] if t["speaker"] == "sys")
total = sum(strat.values())
for s, n in strat.most_common():
    print(f"{s:30s} {n:5d}  {100 * n / total:5.1f}%")
print("total supporter turns:", total)

## 2. Metrics (paper Section 4.2)

In [ ]:
import json
import math
from collections import Counter

from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from sklearn.metrics import f1_score



def tokenize(text):
    return text.lower().replace(".", " .").replace(",", " ,").replace("?", " ?").replace("!", " !").split()


def distinct(hyps, n):
    grams = [tuple(h[i:i + n]) for h in hyps for i in range(len(h) - n + 1)]
    return 100.0 * len(set(grams)) / max(len(grams), 1)


def lcs(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]


def rouge_l(hyp, ref, beta=1.2):
    if not hyp or not ref:
        return 0.0
    l = lcs(hyp, ref)
    if l == 0:
        return 0.0
    r, p = l / len(ref), l / len(hyp)
    return (1 + beta ** 2) * r * p / (r + beta ** 2 * p)


def generation_metrics(refs, hyps):
    R = [[tokenize(r)] for r in refs]
    H = [tokenize(h) for h in hyps]
    sm = SmoothingFunction().method1
    return {
        "B-2": 100 * corpus_bleu(R, H, weights=(0.5, 0.5), smoothing_function=sm),
        "B-4": 100 * corpus_bleu(R, H, weights=(0.25,) * 4, smoothing_function=sm),
        "D-1": distinct(H, 1),
        "D-2": distinct(H, 2),
        "R-L": 100 * sum(rouge_l(h, r[0]) for h, r in zip(H, R)) / len(H),
    }


def strategy_metrics(gold, pred):
    acc = 100 * sum(g == p for g, p in zip(gold, pred)) / len(gold)
    per = f1_score(gold, pred, labels=STRATEGIES, average=None, zero_division=0)
    counts = Counter(pred)
    dist = [counts[s] / len(pred) for s in STRATEGIES]
    ent = -sum(p * math.log2(p) for p in dist if p > 0)
    uni = [1 / len(STRATEGIES)] * len(STRATEGIES)
    m = [(p + u) / 2 for p, u in zip(dist, uni)]
    kl = lambda a, b: sum(x * math.log2(x / y) for x, y in zip(a, b) if x > 0)
    return {
        "ACC": acc,
        "Macro-F1": 100 * per.mean(),
        "Entropy": ent,
        "Bias(JSD)": 0.5 * kl(dist, m) + 0.5 * kl(uni, m),
        "per_strategy_F1": {s: round(100 * f, 2) for s, f in zip(STRATEGIES, per)},
    }


def retrieval_metrics(ranked_lists, golds, ks=(1, 5, 10)):
    out = {f"R@{k}": 0.0 for k in ks}
    mrr = ndcg = 0.0
    for ranked, g in zip(ranked_lists, golds):
        rank = ranked.index(g) + 1 if g in ranked else None
        for k in ks:
            out[f"R@{k}"] += rank is not None and rank <= k
        if rank and rank <= 10:
            mrr += 1 / rank
            ndcg += 1 / math.log2(rank + 1)  # one relevant item, so IDCG = 1
    n = len(golds)
    out = {k: 100 * v / n for k, v in out.items()}
    out["MRR@10"] = mrr / n
    out["nDCG@10"] = ndcg / n
    return out


def evaluate_file(path):
    rows = [json.loads(l) for l in open(path)]
    res = {"n": len(rows)}
    if all("prediction" in r for r in rows):
        res.update(generation_metrics([r["response"] for r in rows], [r["prediction"] for r in rows]))
    if all("pred_strategy" in r for r in rows):
        res.update(strategy_metrics([r["strategy"] for r in rows], [r["pred_strategy"] for r in rows]))
    if all("ranked_ids" in r for r in rows):
        res.update(retrieval_metrics([r["ranked_ids"] for r in rows], [r["gold_id"] for r in rows]))
    return res


## 3. Shared helpers

In [ ]:
import json
import math
import os
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (AutoTokenizer, T5ForConditionalGeneration,
                          get_cosine_schedule_with_warmup, get_linear_schedule_with_warmup)
from transformers.modeling_outputs import BaseModelOutput


DEV = "cuda" if torch.cuda.is_available() else "cpu"
# bf16 autocast only where the GPU supports it (not on Colab T4); T5 is unstable in fp16, so fall back to fp32.
AMP = DEV == "cuda" and torch.cuda.is_bf16_supported()
SPECIAL = ["[STRAT]", "[RESP]", "[CAND]", "[CTX]"]
COMET_RELS = ["xIntent", "xNeed", "xEffect", "xWant"]


# ---------------------------------------------------------------- utilities
def load(path):
    return [json.loads(l) for l in open(path)]


def seed_all(s):
    random.seed(s)
    torch.manual_seed(s)


def ctx_text(ex, turns=6):
    # ASSUMPTION: last 6 utterances of context, speaker-tagged.
    return " ".join(f"{'Seeker' if t['speaker'] == 'usr' else 'Supporter'}: {t['text']}" for t in ex["context"][-turns:])


def target_text(strategy, response, use_strategy=True):
    return f"[STRAT] {strategy} [RESP] {response}" if use_strategy else f"[RESP] {response}"


def get_tok(path):
    tok = AutoTokenizer.from_pretrained(path)
    tok.add_special_tokens({"additional_special_tokens": SPECIAL})
    return tok


def batches(xs, n):
    for i in range(0, len(xs), n):
        yield xs[i:i + n]


# ------------------------------------------------- stage 1: self-retrieval LM
def selfret_pairs(train, use_strategy, neg_ratio=1.0):
    pairs = [("context: " + ctx_text(e), target_text(e["strategy"], e["response"], use_strategy)) for e in train]
    # Self-evaluation task: real next response -> yes, response from another dialogue -> no.
    # ASSUMPTION: this is how P_reject (Sec. 3.4) is learned; the paper does not say.
    for e in train:
        pairs.append((f"judge: {ctx_text(e)} response: {e['response']}", "yes"))
        if random.random() < neg_ratio:
            o = random.choice(train)
            while o["dialog_idx"] == e["dialog_idx"]:
                o = random.choice(train)
            pairs.append((f"judge: {ctx_text(e)} response: {o['response']}", "no"))
    random.shuffle(pairs)
    return pairs


def train_seq2seq(model, tok, pairs, out, epochs, lr, bs, warmup, sched="linear", grad_accum=1, fp16=True):
    model.to(DEV).train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * math.ceil(len(pairs) / bs) // grad_accum
    mk = get_cosine_schedule_with_warmup if sched == "cosine" else get_linear_schedule_with_warmup
    sch = mk(opt, int(warmup * steps) if warmup < 1 else int(warmup), steps)
    use_amp = fp16 and AMP
    for ep in range(epochs):
        random.shuffle(pairs)
        tot = 0.0
        for i, b in enumerate(batches(pairs, bs)):
            enc = tok([x for x, _ in b], max_length=256, truncation=True, padding=True, return_tensors="pt").to(DEV)
            lab = tok([y for _, y in b], max_length=128, truncation=True, padding=True, return_tensors="pt").input_ids.to(DEV)
            lab[lab == tok.pad_token_id] = -100
            with torch.autocast(DEV, dtype=torch.bfloat16, enabled=use_amp):
                loss = model(**enc, labels=lab).loss / grad_accum
            loss.backward()
            if (i + 1) % grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(), sch.step(), opt.zero_grad()
            tot += loss.item() * grad_accum
        print(f"epoch {ep + 1}/{epochs} loss {tot / (i + 1):.4f}", flush=True)
        model.save_pretrained(out)
        tok.save_pretrained(out)


def cmd_train_selfret(a):
    seed_all(a.seed)
    tok = get_tok(a.base)
    model = T5ForConditionalGeneration.from_pretrained(a.base)
    model.resize_token_embeddings(len(tok))
    train = load(f"data/{a.split}/train.jsonl")
    pairs = selfret_pairs(train, not a.no_strategy_token)
    # Table B.11 "SFT" column: lr 2e-5, bs 16, 3 epochs, warmup 0.03, cosine. ASSUMPTION: T5 needs a
    # higher lr than an LLM; default 1e-4 (the "Retrieval" column), override with --lr.
    train_seq2seq(model, tok, pairs, a.out, a.epochs, a.lr, a.bs, 0.03, "cosine")


# ------------------------------------------------------ stage 2: RQ-VAE ResIDs
@torch.no_grad()
def embed(model, tok, texts, bs=64):
    model.to(DEV).eval()
    out = []
    for b in batches(texts, bs):
        enc = tok(b, max_length=128, truncation=True, padding=True, return_tensors="pt").to(DEV)
        h = model.encoder(**enc).last_hidden_state
        m = enc.attention_mask.unsqueeze(-1).float()
        out.append(F.normalize((h * m).sum(1) / m.sum(1), dim=-1).float().cpu())
    return torch.cat(out)


def sinkhorn(cost, eps=0.1, iters=10):
    q = torch.exp(-(cost - cost.min()) / (eps * cost.std().clamp_min(1e-6)))
    q /= q.sum()
    n, k = q.shape
    for _ in range(iters):
        q /= q.sum(0, keepdim=True) * k
        q /= q.sum(1, keepdim=True) * n
    return q.argmax(1)


class RQVAE(nn.Module):
    """4-level residual quantiser (B.2.1). ASSUMPTION: an MLP encoder/decoder
    over sentence vectors; the paper's 6-layer Transformer has no stated input."""

    def __init__(self, d_in=768, d_code=256, levels=4, size=512):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(d_in, 512), nn.GELU(), nn.Linear(512, d_code))
        self.dec = nn.Sequential(nn.Linear(d_code, 512), nn.GELU(), nn.Linear(512, d_in))
        self.books = nn.ParameterList([nn.Parameter(torch.randn(size, d_code) * 0.1) for _ in range(levels)])

    def quantise(self, z, balance=False):
        codes, zq, r, vq, commit = [], 0, z, 0, 0
        for book in self.books:
            cost = torch.cdist(r, book) ** 2
            idx = sinkhorn(cost.detach()) if balance else cost.argmin(1)
            e = book[idx]
            vq = vq + F.mse_loss(e, r.detach())
            commit = commit + F.mse_loss(r, e.detach())
            codes.append(idx)
            zq = zq + e
            r = r - e.detach()
        return torch.stack(codes, 1), zq, vq, commit

    def forward(self, x):
        z = self.enc(x)
        codes, zq, vq, commit = self.quantise(z, balance=self.training)
        zq = z + (zq - z).detach()  # straight-through
        return codes, F.mse_loss(self.dec(zq), x) + 0.25 * vq + 0.25 * commit  # B.2.1 lambdas

    @torch.no_grad()
    def codes(self, x):
        return self.quantise(self.enc(x))[0]


def cmd_build_index(a):
    seed_all(a.seed)
    tok = get_tok(a.model)
    model = T5ForConditionalGeneration.from_pretrained(a.model)
    train = load(f"data/{a.split}/train.jsonl")
    R = embed(model, tok, [e["response"] for e in train])
    rq = RQVAE(d_in=R.shape[1]).to(DEV)
    opt = torch.optim.AdamW(rq.parameters(), lr=1e-4, weight_decay=0.01)
    steps = a.epochs * math.ceil(len(R) / 256)
    sch = get_cosine_schedule_with_warmup(opt, min(2000, steps // 10), steps)
    # Initialise codebooks from data (k-means++-lite) to avoid dead codes.
    with torch.no_grad():
        z = rq.enc(R.to(DEV))
        for book in rq.books:
            book.copy_(z[torch.randint(len(z), (book.shape[0],))] + 1e-3 * torch.randn_like(book))
            z = z - book[torch.cdist(z, book).argmin(1)]
    for ep in range(a.epochs):
        rq.train()
        perm = torch.randperm(len(R))
        for b in batches(perm, 256):
            _, loss = rq(R[b].to(DEV))
            opt.zero_grad(), loss.backward()
            torch.nn.utils.clip_grad_norm_(rq.parameters(), 1.0)
            opt.step(), sch.step()
        if (ep + 1) % 10 == 0:
            print(f"rqvae epoch {ep + 1} loss {loss.item():.4f}", flush=True)
    rq.eval()
    codes = rq.codes(R.to(DEV)).cpu()
    used = [len(set(codes[:, m].tolist())) for m in range(codes.shape[1])]
    uniq = len(set(map(tuple, codes.tolist())))
    print(f"codes used per level {used}; unique ResIDs {uniq}/{len(codes)}")
    os.makedirs(a.out, exist_ok=True)
    torch.save({"rq": rq.state_dict(), "codes": codes, "emb": R, "d_in": R.shape[1]}, os.path.join(a.out, "index.pt"))


# --------------------------------------------------- stage 3: COMET and HEAL
def cmd_knowledge(a):
    """Cache COMET inferences and HEAL top-K expert responses per example.
    --comet: a COMET-ATOMIC-2020 BART checkpoint dir (ASSUMPTION on version).
    --heal:  JSONL of {"text": expert response} from the HEAL graph."""
    from transformers import AutoModelForSeq2SeqLM
    sent_tok = get_tok(a.model)
    sent = T5ForConditionalGeneration.from_pretrained(a.model)
    heal = [json.loads(l)["text"] for l in open(a.heal)] if a.heal else []
    heal_emb = embed(sent, sent_tok, heal) if heal else None
    if a.comet:
        ctok = AutoTokenizer.from_pretrained(a.comet)
        cmodel = AutoModelForSeq2SeqLM.from_pretrained(a.comet).to(DEV).eval()
    for split in ("train", "valid", "test"):
        rows = load(f"data/{a.split}/{split}.jsonl")
        out = []
        for b in batches(rows, 32):
            comet = [[] for _ in b]
            if a.comet:
                for rel in COMET_RELS:
                    enc = ctok([f"{e['query']} {rel} [GEN]" for e in b], return_tensors="pt", padding=True,
                               truncation=True, max_length=64).to(DEV)
                    with torch.no_grad():
                        gen = cmodel.generate(**enc, num_beams=1, max_new_tokens=16)
                    for i, t in enumerate(ctok.batch_decode(gen, skip_special_tokens=True)):
                        comet[i].append(f"{rel} {t.strip()}")
            heal_k = [[] for _ in b]
            if heal:
                q = embed(sent, sent_tok, [e["query"] for e in b])
                top = (q @ heal_emb.T).topk(a.k, dim=1).indices
                heal_k = [[heal[j] for j in row] for row in top.tolist()]
            for e, c, h in zip(b, comet, heal_k):
                out.append({"id": e["id"], "comet": c, "heal": h})
        os.makedirs(a.out, exist_ok=True)
        with open(os.path.join(a.out, f"{split}.jsonl"), "w") as f:
            for r in out:
                f.write(json.dumps(r) + "\n")
        print(split, len(out), flush=True)


def knowledge_map(kdir, split):
    p = os.path.join(kdir, f"{split}.jsonl") if kdir else None
    return {r["id"]: r for r in load(p)} if p and os.path.exists(p) else {}


# ------------------------------------------------ stage 4: fusion generator
class FusionT5(nn.Module):
    """Eq. 6: H_out = l1 H_ctx + l2 CA(H_ctx, H_C) + l3 CA(H_ctx, H_H), l softmax-normalised."""

    def __init__(self, t5):
        super().__init__()
        self.t5 = t5
        d = t5.config.d_model
        self.ca_c = nn.MultiheadAttention(d, 8, batch_first=True)
        self.ca_h = nn.MultiheadAttention(d, 8, batch_first=True)
        self.lam = nn.Parameter(torch.tensor([2.0, 0.0, 0.0]))

    def fused(self, ctx, comet, heal):
        enc = self.t5.encoder
        H = enc(**ctx).last_hidden_state
        HC = enc(**comet).last_hidden_state
        HH = enc(**heal).last_hidden_state
        l1, l2, l3 = F.softmax(self.lam, 0)
        c = self.ca_c(H, HC, HC, key_padding_mask=comet["attention_mask"] == 0)[0]
        h = self.ca_h(H, HH, HH, key_padding_mask=heal["attention_mask"] == 0)[0]
        return BaseModelOutput(last_hidden_state=l1 * H + l2 * c + l3 * h)

    def forward(self, ctx, comet, heal, labels):
        return self.t5(encoder_outputs=self.fused(ctx, comet, heal), attention_mask=ctx["attention_mask"], labels=labels)

    def generate(self, ctx, comet, heal, **kw):
        return self.t5.generate(encoder_outputs=self.fused(ctx, comet, heal), attention_mask=ctx["attention_mask"], **kw)


def fusion_inputs(tok, exs, cands, strategies, kmap, use_comet=True, use_heal=True):
    ctx = [f"[STRAT] {s} [CAND] {c} [CTX] {ctx_text(e)}" for e, c, s in zip(exs, cands, strategies)]
    comet = [" ; ".join(kmap.get(e["id"], {}).get("comet", [])) if use_comet else "" for e in exs]
    heal = [" ; ".join(kmap.get(e["id"], {}).get("heal", [])) if use_heal else "" for e in exs]
    f = lambda xs, n: tok([x or "none" for x in xs], max_length=n, truncation=True, padding=True, return_tensors="pt").to(DEV)
    return f(ctx, 384), f(comet, 128), f(heal, 256)


def load_index(path):
    ix = torch.load(os.path.join(path, "index.pt"))
    rq = RQVAE(d_in=ix["d_in"]).to(DEV)
    rq.load_state_dict(ix["rq"])
    return rq.eval(), ix["codes"], ix["emb"]


def resid_topk(qcode, qemb, codes, emb, mask, r, use_resid=True):
    """Hamming distance on ResIDs (Sec. 3.2); ASSUMPTION: ties broken by cosine."""
    cos = emb @ qemb
    score = -(codes != qcode).sum(1).float() * 10.0 + cos if use_resid else cos
    score[~mask] = -1e9
    return score.topk(min(r, int(mask.sum()))).indices.tolist()


def cmd_train_fusion(a):
    seed_all(a.seed)
    tok = get_tok(a.selfret)
    sr = T5ForConditionalGeneration.from_pretrained(a.selfret)
    train = load(f"data/{a.split}/train.jsonl")
    rq, codes, emb = load_index(a.index)
    # Candidate for training = best ResID neighbour with the gold strategy from another dialogue.
    q = embed(sr, tok, ["context: " + ctx_text(e) for e in train])
    qcodes = rq.codes(q.to(DEV)).cpu()
    strat = [e["strategy"] for e in train]
    dlg = torch.tensor([e["dialog_idx"] for e in train])
    cands = []
    for i, e in enumerate(train):
        mask = torch.tensor([s == e["strategy"] for s in strat]) & (dlg != e["dialog_idx"])
        cands.append(train[resid_topk(qcodes[i], q[i], codes, emb, mask, 1)[0]]["response"])
    kmap = knowledge_map(a.knowledge, "train")
    t5 = T5ForConditionalGeneration.from_pretrained(a.base)
    t5.resize_token_embeddings(len(tok))
    model = FusionT5(t5).to(DEV).train()
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=0.01)
    idx = list(range(len(train)))
    steps = a.epochs * math.ceil(len(idx) / a.bs)
    sch = get_linear_schedule_with_warmup(opt, 100, steps)
    for ep in range(a.epochs):
        random.shuffle(idx)
        tot = 0
        for n, b in enumerate(batches(idx, a.bs)):
            exs = [train[i] for i in b]
            ctx, com, hel = fusion_inputs(tok, exs, [cands[i] for i in b], [e["strategy"] for e in exs], kmap)
            lab = tok([e["response"] for e in exs], max_length=128, truncation=True, padding=True, return_tensors="pt").input_ids.to(DEV)
            lab[lab == tok.pad_token_id] = -100
            with torch.autocast(DEV, dtype=torch.bfloat16, enabled=AMP):
                loss = model(ctx, com, hel, lab).loss
            opt.zero_grad(), loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(), sch.step()
            tot += loss.item()
        print(f"fusion epoch {ep + 1} loss {tot / (n + 1):.4f} lambda {F.softmax(model.lam, 0).tolist()}", flush=True)
    os.makedirs(a.out, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(a.out, "fusion.pt"))
    tok.save_pretrained(a.out)
    json.dump({"base": a.base}, open(os.path.join(a.out, "config.json"), "w"))


# ------------------------------------------------------ stage 5: inference
@torch.no_grad()
def strategy_logprobs(model, tok, src):
    enc = tok([src] * len(STRATEGIES), max_length=256, truncation=True, return_tensors="pt", padding=True).to(DEV)
    lab = tok([f"[STRAT] {s} [RESP]" for s in STRATEGIES], return_tensors="pt", padding=True).input_ids.to(DEV)
    lab[lab == tok.pad_token_id] = -100
    logits = model(**enc, labels=lab).logits
    lp = -F.cross_entropy(logits.transpose(1, 2), lab, reduction="none", ignore_index=-100).sum(1)
    return lp.cpu()


class Trie:
    def __init__(self, seqs):
        self.root = {}
        for s in seqs:
            node = self.root
            for t in s:
                node = node.setdefault(t, {})

    def next(self, prefix):
        node = self.root
        for t in prefix:
            node = node.get(t)
            if node is None:
                return []
        return list(node.keys())


@torch.no_grad()
def constrained_beam(model, tok, src, targets, beams):
    """Eq. 4: decode only along the prefix trie of BPE-tokenised candidates."""
    seqs = [tok(t, add_special_tokens=True).input_ids for t in targets]  # ends with </s>
    trie = Trie(seqs)
    start = model.config.decoder_start_token_id
    enc = tok(src, max_length=256, truncation=True, return_tensors="pt").to(DEV)

    def allowed(_, ids):
        nxt = trie.next(ids.tolist()[1:])  # drop decoder start token
        return nxt or [tok.eos_token_id]

    out = model.generate(**enc, num_beams=beams, num_return_sequences=min(beams, len(targets)),
                         prefix_allowed_tokens_fn=allowed, max_new_tokens=160, output_scores=True,
                         return_dict_in_generate=True, length_penalty=1.0, decoder_start_token_id=start)
    texts = tok.batch_decode(out.sequences, skip_special_tokens=False)
    clean = [t.replace("<pad>", "").replace("</s>", "").strip() for t in texts]
    return list(zip(clean, out.sequences_scores.cpu().tolist()))


def split_target(t):
    if "[RESP]" in t:
        head, resp = t.split("[RESP]", 1)
        return head.replace("[STRAT]", "").strip(), resp.strip()
    return None, t.strip()


@torch.no_grad()
def p_valid(model, tok, ctx, responses):
    src = [f"judge: {ctx} response: {r}" for r in responses]
    enc = tok(src, max_length=256, truncation=True, padding=True, return_tensors="pt").to(DEV)
    start = torch.full((len(src), 1), model.config.decoder_start_token_id, device=DEV)
    logits = model(**enc, decoder_input_ids=start).logits[:, 0]
    yes, no = tok("yes", add_special_tokens=False).input_ids[0], tok("no", add_special_tokens=False).input_ids[0]
    return F.softmax(logits[:, [yes, no]], -1)[:, 0].cpu()


class SageModel:
    """Loaded SAGE pipeline; respond() runs Sec. 3.3-3.5 for one dialogue context."""

    def __init__(self, a):
        self.a = a
        self.tok = get_tok(a.selfret)
        self.sr = T5ForConditionalGeneration.from_pretrained(a.selfret).to(DEV).eval()
        self.rq, self.codes, self.emb = load_index(a.index)
        self.train = load(f"data/{a.split}/train.jsonl")
        self.strat_of = [e["strategy"] for e in self.train]
        self.kmap = knowledge_map(a.knowledge, a.eval_split)
        self.fusion = None
        if a.fusion and not a.retrieval_only:
            cfg = json.load(open(os.path.join(a.fusion, "config.json")))
            t5 = T5ForConditionalGeneration.from_pretrained(cfg["base"])
            t5.resize_token_embeddings(len(self.tok))
            self.fusion = FusionT5(t5)
            self.fusion.load_state_dict(torch.load(os.path.join(a.fusion, "fusion.pt"), map_location="cpu"))
            self.fusion.to(DEV).eval()

    @torch.no_grad()
    def respond(self, e):
        a, tok, sr, train = self.a, self.tok, self.sr, self.train
        src = "context: " + ctx_text(e)
        q = embed(sr, tok, [src])[0]
        qcode = self.rq.codes(q.unsqueeze(0).to(DEV))[0].cpu()
        sprob = F.softmax(strategy_logprobs(sr, tok, src) / a.tau, 0)
        top_s = [STRATEGIES[j] for j in sprob.topk(a.bs_strat).indices.tolist()]
        cands = []  # (strategy, response, strategy prob)
        for s in top_s:
            mask = torch.tensor([x == s for x in self.strat_of])
            ids = resid_topk(qcode, q, self.codes, self.emb, mask, a.r, use_resid=not a.no_resid)
            if a.no_constraint:
                enc = tok(src, return_tensors="pt", max_length=256, truncation=True).to(DEV)
                dec = tok(f"[STRAT] {s} [RESP]", add_special_tokens=False, return_tensors="pt").input_ids.to(DEV)
                start = torch.full((1, 1), sr.config.decoder_start_token_id, device=DEV)
                gen = sr.generate(**enc, decoder_input_ids=torch.cat([start, dec], 1), num_beams=a.br,
                                  num_return_sequences=a.br, max_new_tokens=128, no_repeat_ngram_size=3)
                outs = [split_target(t)[1] for t in tok.batch_decode(gen, skip_special_tokens=False)]
                outs = [o.replace("<pad>", "").replace("</s>", "").strip() for o in outs]
            else:
                targets = [target_text(s, train[j]["response"]) for j in ids]
                outs = [split_target(t)[1] for t, _ in constrained_beam(sr, tok, src, targets, a.br)]
            cands += [(s, o, float(sprob[STRATEGIES.index(s)])) for o in outs if o]
        if not cands:  # Sec. 3.3 fallback
            cands = [("Information", train[0]["response"], 1e-6)]
        # Eq. 5, in log space.
        pv = p_valid(sr, tok, ctx_text(e), [c[1] for c in cands])
        log_s = torch.tensor([math.log(c[2] + 1e-12) for c in cands]) + torch.log(pv + 1e-12) / a.delta
        kn = self.kmap.get(e.get("id"), {})
        cemb = embed(sr, tok, [c[1] for c in cands])
        if kn.get("comet") and not a.no_comet:
            ce = embed(sr, tok, kn["comet"])
            log_s += a.alpha * torch.log((cemb @ ce.T).mean(1).clamp_min(1e-6))
        if kn.get("heal") and not a.no_heal:
            he = embed(sr, tok, kn["heal"])
            log_s += a.beta * torch.log((cemb @ he.T).max(1).values.clamp_min(1e-6))
        best = int(log_s.argmax())
        s_star, r_star = cands[best][0], cands[best][1]
        pred = r_star
        if self.fusion is not None:
            ctx, com, hel = fusion_inputs(tok, [e], [r_star], [s_star], self.kmap, not a.no_comet, not a.no_heal)
            gen = self.fusion.generate(ctx, com, hel, num_beams=4, max_new_tokens=128, min_new_tokens=10,
                                       repetition_penalty=1.2, no_repeat_ngram_size=3, length_penalty=1.0)
            pred = tok.decode(gen[0], skip_special_tokens=True).strip()
        return {"pred_strategy": top_s[0], "chosen_strategy": s_star, "candidate": r_star, "prediction": pred}

    @torch.no_grad()
    def gold_nll(self, e, r_star):
        """PPL term: gold response under the fusion generator, given the gold strategy (ASSUMPTION)."""
        tok, a = self.tok, self.a
        ctx, com, hel = fusion_inputs(tok, [e], [r_star], [e["strategy"]], self.kmap, not a.no_comet, not a.no_heal)
        lab = tok([e["response"]], return_tensors="pt", max_length=128, truncation=True).input_ids.to(DEV)
        return self.fusion(ctx, com, hel, lab).loss.item() * lab.shape[1], lab.shape[1]


def cmd_infer(a):
    seed_all(a.seed)
    model = SageModel(a)
    test = load(f"data/{a.split}/{a.eval_split}.jsonl")
    if a.limit:
        test = test[:a.limit]
    os.makedirs(a.out, exist_ok=True)
    f = open(os.path.join(a.out, "predictions.jsonl"), "w")
    nll_sum = ntok = 0
    for i, e in enumerate(test):
        out = model.respond(e)
        if model.fusion is not None:
            nll, n = model.gold_nll(e, out["candidate"])
            nll_sum, ntok = nll_sum + nll, ntok + n
        f.write(json.dumps({"id": e["id"], "response": e["response"], "strategy": e["strategy"], **out}) + "\n")
        if (i + 1) % 50 == 0:
            f.flush()
            print(f"{i + 1}/{len(test)}", flush=True)
    f.close()
    res = evaluate_file(os.path.join(a.out, "predictions.jsonl"))
    if ntok:
        res["PPL"] = math.exp(nll_sum / ntok)
    res["args"] = vars(a)
    json.dump(res, open(os.path.join(a.out, "metrics.json"), "w"), indent=2)
    print(json.dumps({k: v for k, v in res.items() if k != "args"}, indent=2))


def cmd_eval_retrieval(a):
    """Tables 4/8. ASSUMPTION: pool = a random --pool-size subset of test responses
    that always contains the gold one (the paper never states its pool; on the full
    3.5k-response test pool BM25 gets R@10 = 11.4, far below the paper's DPR 61.8)."""
    seed_all(a.seed)
    tok = get_tok(a.selfret)
    sr = T5ForConditionalGeneration.from_pretrained(a.selfret).to(DEV).eval()
    rq, _, _ = load_index(a.index)
    test = load(f"data/{a.split}/test.jsonl")
    if a.limit:
        test = test[:a.limit]
    pemb = embed(sr, tok, [e["response"] for e in test])
    pcodes = rq.codes(pemb.to(DEV)).cpu()
    rows = []
    for i, e in enumerate(test):
        pool = random.sample([j for j in range(len(test)) if j != i], min(a.pool_size - 1, len(test) - 1)) + [i]
        src = "context: " + ctx_text(e)
        q = embed(sr, tok, [src])[0]
        qcode = rq.codes(q.unsqueeze(0).to(DEV))[0].cpu()
        sub = torch.tensor(pool)
        pre = [pool[j] for j in resid_topk(qcode, q, pcodes[sub], pemb[sub], torch.ones(len(pool), dtype=torch.bool), a.r)]
        targets = {target_text(test[j]["strategy"], test[j]["response"]): j for j in pre}
        ranked = constrained_beam(sr, tok, src, list(targets), min(10, len(targets)))
        ids = [targets.get(t, -1) for t, _ in ranked]
        ids = [j for j in ids if j >= 0]
        rows.append({"id": e["id"], "ranked_ids": ids, "gold_id": i, "num_prev_turns": e["num_prev_turns"]})
    res = retrieval_metrics([r["ranked_ids"] for r in rows], [r["gold_id"] for r in rows])
    res["pool_size"] = a.pool_size
    os.makedirs(a.out, exist_ok=True)
    json.dump(res, open(os.path.join(a.out, "retrieval.json"), "w"), indent=2)
    print(json.dumps(res, indent=2))


## 4. BM25 floor (CPU)

This is a no-learning baseline. For retrieval, it searches the full pool of test responses. For generation, it returns the nearest training response. It takes ~15 minutes on the full test set.

In [ ]:
import json
import os

from rank_bm25 import BM25Okapi



def load(path):
    return [json.loads(l) for l in open(path)]


def query_text(ex, turns=3):
    return " ".join(t["text"] for t in ex["context"][-turns:])


def bucket(n_prev):
    # Table 8 buckets count dialogue turns; one turn = a seeker+supporter pair.
    turns = (n_prev + 1) // 2
    return "1-2" if turns <= 2 else "3-4" if turns <= 4 else "5-6" if turns <= 6 else ">=7"


def run_bm25(args):
    train = load(f"data/{args.split}/train.jsonl")
    test = load(f"data/{args.split}/test.jsonl")
    os.makedirs(args.out, exist_ok=True)

    # 1. Retrieval over the test response pool.
    pool = BM25Okapi([tokenize(e["response"]) for e in test])
    ranked = []
    for e in test:
        scores = pool.get_scores(tokenize(query_text(e)))
        ranked.append(sorted(range(len(test)), key=lambda i: -scores[i])[:10])
    golds = list(range(len(test)))
    ret = retrieval_metrics(ranked, golds)
    by_len = {}
    for b in ("1-2", "3-4", "5-6", ">=7"):
        idx = [i for i, e in enumerate(test) if bucket(e["num_prev_turns"]) == b]
        by_len[b] = {"n": len(idx), "R@10": retrieval_metrics([ranked[i] for i in idx], idx)["R@10"]}

    # 2. Nearest training response as the generated reply.
    index = BM25Okapi([tokenize(query_text(e)) for e in train])
    preds = []
    for e in test:
        scores = index.get_scores(tokenize(query_text(e)))
        best = train[max(range(len(train)), key=lambda i: scores[i])]
        preds.append({"id": e["id"], "response": e["response"], "strategy": e["strategy"],
                      "prediction": best["response"], "pred_strategy": best["strategy"]})
    with open(os.path.join(args.out, "predictions.jsonl"), "w") as f:
        for p in preds:
            f.write(json.dumps(p) + "\n")

    gen = generation_metrics([p["response"] for p in preds], [p["prediction"] for p in preds])
    strat = strategy_metrics([p["strategy"] for p in preds], [p["pred_strategy"] for p in preds])
    res = {"split": args.split, "n_test": len(test), "retrieval": ret, "retrieval_R@10_by_turns": by_len,
           "generation": gen, "strategy": strat}
    json.dump(res, open(os.path.join(args.out, "metrics.json"), "w"), indent=2)
    print(json.dumps(res, indent=2))


In [ ]:
from types import SimpleNamespace
run_bm25(SimpleNamespace(split=SPLIT, out="results/bm25"))

## 5. Open-LLM baselines (Tables 1-3)

The prompt follows Section 4.4: the dialogue context plus descriptions of the 8 strategies, and the model picks a strategy and then responds. PPL is the gold response's perplexity given the prompt and the gold strategy. Runs resume if interrupted.

In [ ]:
import json
import math
import os
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


STRATEGY_DESC = {
    "Question": "Ask for information related to the problem to help the help-seeker articulate their issues.",
    "Restatement or Paraphrasing": "A simple, more concise rephrasing of the help-seeker's statements.",
    "Reflection of feelings": "Articulate and describe the help-seeker's feelings.",
    "Self-disclosure": "Divulge similar experiences you have had or emotions you share, to express empathy.",
    "Affirmation and Reassurance": "Affirm the help-seeker's strengths, motivation, and capabilities, and provide reassurance and encouragement.",
    "Providing Suggestions": "Provide suggestions about how to change, but be careful not to overstep and tell them what to do.",
    "Information": "Provide useful information to the help-seeker, for example with data, facts, opinions, resources, or by answering questions.",
    "Others": "Exchange pleasantries and use other support strategies that do not fall into the above categories.",
}

SYSTEM = (
    "You are a supporter in an emotional support conversation. Choose the single most appropriate "
    "support strategy for your next reply from the list below, then write the reply using that strategy.\n\n"
    + "\n".join(f"- {s}: {d}" for s, d in STRATEGY_DESC.items())
    + "\n\nAnswer in exactly this format:\nStrategy: <one strategy name from the list>\nResponse: <your reply, one or two sentences>"
)


def user_prompt(ex, max_turns=10):
    lines = [f"Situation: {ex['situation']}", "", "Conversation so far:"]
    for t in ex["context"][-max_turns:]:
        lines.append(f"{'Help-seeker' if t['speaker'] == 'usr' else 'Supporter'}: {t['text']}")
    return "\n".join(lines)


def build_prompt(tok, ex):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_prompt(ex)}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except Exception:  # Mistral templates reject a system role
        msgs = [{"role": "user", "content": SYSTEM + "\n\n" + user_prompt(ex)}]
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def parse(text):
    m = re.search(r"Strategy:\s*(.+)", text)
    raw = m.group(1).strip().lower() if m else ""
    strat = next((s for s in STRATEGIES if s.lower() in raw or raw.startswith(s.split()[0].lower())), "Others")
    r = re.search(r"Response:\s*(.+)", text, re.S)
    resp = (r.group(1) if r else text).strip().split("\n")[0]
    return strat, resp


@torch.no_grad()
def gold_nll(model, tok, prompt, ex):
    target = f"Strategy: {ex['strategy']}\nResponse: "
    full = prompt + target + ex["response"]
    pre = tok(prompt + target, return_tensors="pt", add_special_tokens=False).input_ids
    ids = tok(full, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
    labels = ids.clone()
    labels[:, : pre.shape[1]] = -100
    loss = model(input_ids=ids, labels=labels).loss
    n = int((labels != -100).sum())
    return float(loss) * n, n


def run_llm(args):

    test = [json.loads(l) for l in open(f"data/{args.split}/test.jsonl")]
    if args.limit:
        test = test[: args.limit]
    os.makedirs(args.out, exist_ok=True)

    tok = AutoTokenizer.from_pretrained(args.model)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    kw = {"torch_dtype": dtype, "device_map": "auto"}
    if args.load_4bit:
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=dtype)
    model = AutoModelForCausalLM.from_pretrained(args.model, **kw).eval()

    pred_path = os.path.join(args.out, "predictions.jsonl")
    done = {json.loads(l)["id"] for l in open(pred_path)} if os.path.exists(pred_path) else set()
    todo = [e for e in test if e["id"] not in done]
    with open(pred_path, "a") as f:
        for i in range(0, len(todo), args.batch_size):
            batch = todo[i : i + args.batch_size]
            prompts = [build_prompt(tok, e) for e in batch]
            enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            out = model.generate(**enc, max_new_tokens=args.max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
            texts = tok.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
            for e, p, t in zip(batch, prompts, texts):
                strat, resp = parse(t)
                row = {"id": e["id"], "response": e["response"], "strategy": e["strategy"],
                       "prediction": resp, "pred_strategy": strat, "raw": t}
                if not args.no_ppl:
                    row["nll"], row["ntok"] = gold_nll(model, tok, p, e)
                f.write(json.dumps(row) + "\n")
            f.flush()
            print(f"{min(i + args.batch_size, len(todo))}/{len(todo)}", flush=True)

    rows = [json.loads(l) for l in open(pred_path)]
    res = {"model": args.model, "split": args.split, "n": len(rows)}
    res.update(generation_metrics([r["response"] for r in rows], [r["prediction"] for r in rows]))
    res.update(strategy_metrics([r["strategy"] for r in rows], [r["pred_strategy"] for r in rows]))
    if all("nll" in r for r in rows):
        res["PPL"] = math.exp(sum(r["nll"] for r in rows) / sum(r["ntok"] for r in rows))
    json.dump(res, open(os.path.join(args.out, "metrics.json"), "w"), indent=2)
    print(json.dumps(res, indent=2))


In [ ]:
import gc
for model_id in LLMS:
    out = "results/" + model_id.split("/")[-1].lower()
    run_llm(SimpleNamespace(model=model_id, split=SPLIT, out=out, limit=LIMIT, batch_size=8,
                            max_new_tokens=64, load_4bit=LOAD_4BIT, no_ppl=False))
    gc.collect(); torch.cuda.empty_cache()

## 6. SAGE re-implementation

1. **Self-retrieval LM** (Eq. 1). One T5 learns `context -> [STRAT] s [RESP] r` together with a yes/no self-evaluation task.
2. **ResIDs**. A 4×512 RQ-VAE with Sinkhorn balancing quantises response embeddings (Section 3.2, Appendix B.2.1).
3. **Fusion generator** (Eq. 6). It cross-attends the context to COMET and HEAL knowledge. Both sources are optional and are skipped here (`knowledge="none"`).
4. **Inference**. It runs strategy top-B_s, then ResID retrieval, then trie-constrained beam search (Eq. 4), then re-ranking (Eq. 5), then fused generation.

In [ ]:
A = SimpleNamespace
cmd_train_selfret(A(split=SPLIT, seed=42, base=T5_BASE, out="ckpt/selfret", epochs=EPOCHS, lr=1e-4, bs=16,
                    no_strategy_token=False))
cmd_build_index(A(split=SPLIT, seed=42, model="ckpt/selfret", out="ckpt/index", epochs=10 if QUICK else 50))
cmd_train_fusion(A(split=SPLIT, seed=42, base=T5_BASE, selfret="ckpt/selfret", index="ckpt/index",
                   knowledge="none", out="ckpt/fusion", epochs=EPOCHS, lr=1e-4, bs=16))

In [ ]:
INFER = dict(split=SPLIT, seed=42, selfret="ckpt/selfret", index="ckpt/index", fusion="ckpt/fusion",
             knowledge="none", eval_split="test", limit=LIMIT, bs_strat=3, r=20, br=4, tau=1.0, delta=1.0,
             alpha=0.5, beta=0.5, no_constraint=False, no_resid=False, no_comet=False, no_heal=False,
             retrieval_only=False)
cmd_infer(A(**INFER, out="results/sage"))
cmd_eval_retrieval(A(split=SPLIT, seed=42, selfret="ckpt/selfret", index="ckpt/index", out="results/sage",
                     pool_size=100, r=20, limit=LIMIT))

### Ablations (Table 6)

In [ ]:
for flag in ("no_constraint", "no_resid", "retrieval_only"):
    cmd_infer(A(**{**INFER, flag: True}, out=f"results/sage_{flag}"))

## 7. ESC-Eval: the second dataset (Table 5, Table C.15)

SAGE evaluates on two datasets. **ESConv** (sections 1-6) is the training and test corpus. **ESC-Eval** (Zhao et al., EMNLP 2024) is an interactive benchmark.

In ESC-Eval, a role-playing model ("ESC-Role") acts as a help-seeker described by a **role card**. It talks with the system under test for 5 turns. A judge then scores each dialogue on seven 0-4 dimensions: fluency, diversity (expression), empathy, information (suggestions), humanoid, skillful (emotional knowledge) and overall.

- The cards are the 331 English high-quality cards from the official ESC-Eval repo, in `data/esceval/`. The 2,801-card pool that SAGE cites is not public.
- The seeker is `haidequanbu/ESC-Role`, a 14B model loaded in 4-bit. If it doesn't fit your GPU, set `SEEKER` to an instruct model such as Qwen2.5-7B, and report that change.
- The judge is ESC-RANK (InternLM2-chat-7B plus a LoRA per dimension). If it fails to load, set `JUDGE = "llm:<model id>"` to use an LLM judge with the same prompts, and report that too.
- SAGE's numbers come from **human** raters on a 5-point scale. ESC-RANK is an automatic judge on a 0-4 scale, so we also show the +1-shifted values, which are only roughly comparable.

In [ ]:
import json
import os
import re
from types import SimpleNamespace

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

CARDS = "data/esceval/card_high_en.json"
PROMPTS = "data/esceval/escrank_prompts_en.json"
SEEKER_PROMPT = ("I hope you can role-play as a person with worries and communicate with an AI assistant. "
                 "Here is your character card:'{card}\n' Please simulate a natural human conversation as much as "
                 "possible. Avoid behaving like a machine and refrain from mentioning that you are an AI assistant. "
                 "Gradually reveal your worries throughout our multiple conversations.")
OPENING = "Hello, I'm your personal assistant. You can confide in me about any worries or concerns you may have!"
DIMENSIONS = ["fluecy", "diversity", "empathy", "suggection effectiveness", "humanoid", "emotional knowledge",
              "human preference"]  # spelled as in ESC-Eval score.py, since ESC-RANK was trained on these prompts
KEYS = ["fluency", "diversity", "empathic", "suggestion", "human", "tech", "overall"]
# SAGE Table 5 / C.15 column -> ESC-Eval dimension.
SAGE_COLS = {"Flu.": "fluency", "Exp.": "diversity", "Emp.": "empathic", "Info.": "suggestion",
             "Hum.": "human", "Skill.": "tech", "Overall": "overall"}


class Chat:
    """Minimal greedy chat wrapper around a Hugging Face causal LM."""

    def __init__(self, model_id, load_4bit=False, max_new_tokens=256, adapters=None, trust_remote_code=False):
        self.tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=trust_remote_code)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        cuda = torch.cuda.is_available()
        dtype = torch.bfloat16 if cuda and torch.cuda.is_bf16_supported() else torch.float16 if cuda else torch.float32
        kw = {"torch_dtype": dtype, "device_map": "auto" if cuda else None, "trust_remote_code": trust_remote_code}
        if load_4bit and cuda:
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=dtype)
        self.model = AutoModelForCausalLM.from_pretrained(model_id, **kw).eval()
        if adapters:  # {name: path}, one LoRA per ESC-RANK dimension
            from peft import PeftModel
            names = list(adapters)
            self.model = PeftModel.from_pretrained(self.model, adapters[names[0]], adapter_name=names[0])
            for n in names[1:]:
                self.model.load_adapter(adapters[n], adapter_name=n)
        self.max_new_tokens = max_new_tokens

    def use(self, adapter):
        self.model.set_adapter(adapter)

    @torch.no_grad()
    def __call__(self, messages):
        try:
            text = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:  # templates without a system role (e.g. Mistral): fold it into the first user turn
            sys_msg = [m["content"] for m in messages if m["role"] == "system"]
            rest = [dict(m) for m in messages if m["role"] != "system"]
            if sys_msg and rest:
                rest[0]["content"] = sys_msg[0] + "\n\n" + rest[0]["content"]
            text = self.tok.apply_chat_template(rest, tokenize=False, add_generation_prompt=True)
        enc = self.tok(text, return_tensors="pt", add_special_tokens=False).to(self.model.device)
        out = self.model.generate(**enc, max_new_tokens=self.max_new_tokens, do_sample=False,
                                  pad_token_id=self.tok.pad_token_id)
        return self.tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True).strip()

    def free(self):
        del self.model
        torch.cuda.empty_cache()


def _alternate(messages):
    """Chat templates such as Mistral's require user/assistant turns to alternate; the seeker
    sees the ESC-Eval opening as a user turn, so merge any consecutive same-role turns."""
    out = []
    for m in messages:
        if out and out[-1]["role"] == m["role"] and m["role"] != "system":
            out[-1] = {"role": m["role"], "content": out[-1]["content"] + "\n" + m["content"]}
        else:
            out.append(dict(m))
    return out


def make_system(spec, args):
    """Return reply(history) -> str, where history is [(speaker, text)] with speaker in {usr, sys}."""
    if spec == "sage":
        sage_args = SimpleNamespace(split=args.split, selfret=args.selfret, index=args.index, fusion=args.fusion,
                                    knowledge="none", eval_split="test", bs_strat=3, r=20, br=4, tau=1.0,
                                    delta=1.0, alpha=0.5, beta=0.5, no_constraint=False, no_resid=False,
                                    no_comet=False, no_heal=False, retrieval_only=False)
        model = SageModel(sage_args)

        def reply(history):
            ex = {"id": None, "context": [{"speaker": s, "text": t} for s, t in history],
                  "query": next(t for s, t in reversed(history) if s == "usr")}
            return model.respond(ex)["prediction"]
        return reply, None

    chat = Chat(spec.split(":", 1)[1], load_4bit=args.load_4bit, max_new_tokens=128)

    def reply(history):
        convo = "\n".join(f"{'Help-seeker' if s == 'usr' else 'Supporter'}: {t}" for s, t in history[-10:])
        out = chat([{"role": "system", "content": SYSTEM}, {"role": "user", "content": "Conversation so far:\n" + convo}])
        return parse(out)[1]
    return reply, chat


def roleplay(args):
    cards = json.load(open(args.cards))
    if args.limit:
        cards = cards[:args.limit]
    os.makedirs(args.out, exist_ok=True)
    path = os.path.join(args.out, "dialogs.json")
    dialogs = json.load(open(path)) if os.path.exists(path) else {}
    seeker = Chat(args.seeker, load_4bit=args.load_4bit, max_new_tokens=args.max_new_tokens)
    system, sys_chat = make_system(args.system, args)
    for i, card in enumerate(cards):
        key = str(card.get("id", i))
        if key in dialogs:
            continue
        seeker_msgs = [{"role": "system", "content": SEEKER_PROMPT.format(card=card["base"])},
                       {"role": "user", "content": OPENING}]
        history = [("sys", OPENING)]
        for _ in range(args.turns):
            said = seeker(_alternate(seeker_msgs))
            seeker_msgs.append({"role": "assistant", "content": said})
            history.append(("usr", said))
            ans = system(history) or "I see."
            seeker_msgs.append({"role": "user", "content": ans})
            history.append(("sys", ans))
        dialogs[key] = {"source": card.get("source"), "card": card["base"],
                        "dialog": [("AI assistant：" if s == "sys" else "ESC-Role：") + t for s, t in history]}
        json.dump(dialogs, open(path, "w"), ensure_ascii=False, indent=1)
        print(f"{i + 1}/{len(cards)}", flush=True)
    seeker.free()
    if sys_chat:
        sys_chat.free()
    return path


def load_judge(args):
    if args.judge == "escrank":
        from huggingface_hub import snapshot_download
        lora = snapshot_download(args.escrank_repo)
        adapters = {k: os.path.join(lora, f"{k}_en") for k in KEYS}
        return Chat(args.escrank_base, load_4bit=args.load_4bit, max_new_tokens=8, adapters=adapters,
                    trust_remote_code=True)
    return Chat(args.judge.split(":", 1)[1], load_4bit=args.load_4bit, max_new_tokens=8)


def score(args):
    dialogs = json.load(open(args.dialogs))
    prompts = json.load(open(PROMPTS))
    judge = load_judge(args)
    out_path = os.path.join(os.path.dirname(args.dialogs), f"scores_{args.judge.replace('/', '_').replace(':', '_')}.json")
    scores = json.load(open(out_path)) if os.path.exists(out_path) else {}
    for n, (key, d) in enumerate(dialogs.items()):
        if key in scores:
            continue
        # Same formatting as ESC-Eval score.py (its role labels included).
        text = "\n\n".join(d["dialog"]).replace("AI assistant", "**AI助手**").replace("ESC-Role：", "**用户**")
        scores[key] = {}
        for p, dim, k in zip(prompts, DIMENSIONS, KEYS):
            if args.judge == "escrank":
                judge.use(k)
            msg = (p + "Dialogue between user and AI assistant: \n" + text.strip() + "\n"
                   + "Based on the rules, give your " + dim + " score (The number only) to the Dialogue.")
            m = re.search(r"[0-4]", judge([{"role": "user", "content": msg}]))
            scores[key][k] = int(m.group()) if m else -1
        json.dump(scores, open(out_path, "w"), indent=1)
        print(f"scored {n + 1}/{len(dialogs)}", flush=True)
    summary = summarise(scores)
    summary.update({"judge": args.judge, "dialogs": args.dialogs})
    json.dump(summary, open(os.path.join(os.path.dirname(args.dialogs), "esceval.json"), "w"), indent=2)
    print(json.dumps(summary, indent=2))
    return summary


def summarise(scores):
    res = {"n": len(scores)}
    for k in KEYS:
        vals = [s[k] for s in scores.values() if s.get(k, -1) >= 0]
        res[k] = sum(vals) / len(vals) if vals else None
    # SAGE-style columns on a 1-5 scale (ESC-Eval 0-4 shifted by +1; ASSUMPTION, see module docstring).
    res["sage_scale_1to5"] = {c: (res[k] + 1 if res[k] is not None else None) for c, k in SAGE_COLS.items()}
    return res


In [ ]:
ESC_LIMIT = 20 if QUICK else 331          # role cards to run (331 = all English high-quality cards)
SEEKER = os.environ.get("SAGE_SEEKER", "haidequanbu/ESC-Role")
JUDGE = os.environ.get("SAGE_JUDGE", "escrank")
ESC_SYSTEMS = ["sage"] + ["llm:" + m for m in LLMS]
for system in ESC_SYSTEMS:
    name = "sage" if system == "sage" else system.split("/")[-1].lower()
    path = roleplay(SimpleNamespace(system=system, seeker=SEEKER, cards=CARDS, out=f"results/esceval_{name}",
                                    turns=5, limit=ESC_LIMIT, max_new_tokens=256, load_4bit=LOAD_4BIT, split=SPLIT,
                                    selfret="ckpt/selfret", index="ckpt/index", fusion="ckpt/fusion"))
    gc.collect(); torch.cuda.empty_cache()
    score(SimpleNamespace(dialogs=path, judge=JUDGE, escrank_repo="haidequanbu/ESC-RANK",
                          escrank_base="internlm/internlm2-chat-7b", load_4bit=LOAD_4BIT))
    gc.collect(); torch.cuda.empty_cache()

## 8. Compare with the paper

In [ ]:
import glob
import json
import os

# SAGE paper, Table 1 (ACC, PPL, B-2, B-4, D-1, D-2, R-L) and Table 2 (Macro-F1).
PAPER = {
    "BlenderBot-Joint": dict(ACC=27.72, PPL=18.11, **{"B-2": 6.18, "B-4": 1.66, "D-1": 3.27, "D-2": 20.87, "R-L": 15.13}),
    "Qwen-2.5-7B": dict(ACC=36.89, PPL=14.58, **{"B-2": 10.89, "B-4": 3.24, "D-1": 8.12, "D-2": 37.86, "R-L": 19.21, "Macro-F1": 29.99}),
    "LLaMA-3.1-8B": dict(ACC=37.45, PPL=15.48, **{"B-2": 9.32, "B-4": 2.76, "D-1": 6.42, "D-2": 33.67, "R-L": 18.34, "Macro-F1": 32.89}),
    "Mistral-7B": dict(ACC=38.91, PPL=15.12, **{"B-2": 9.87, "B-4": 2.94, "D-1": 6.85, "D-2": 35.14, "R-L": 18.92, "Macro-F1": 31.39}),
    "SAGE": dict(ACC=46.80, PPL=14.32, **{"B-2": 11.87, "B-4": 3.88, "D-1": 9.24, "D-2": 39.86, "R-L": 20.51, "Macro-F1": 38.95,
                                            "R@1": 31.6, "R@10": 74.8, "MRR@10": 0.437, "nDCG@10": 0.703}),
}
COLS = ["ACC", "Macro-F1", "PPL", "B-2", "B-4", "D-1", "D-2", "R-L", "R@1", "R@10", "MRR@10", "nDCG@10"]
ESC_PAPER = {  # SAGE Table 5 (human raters, 5-point scale); Exp./Skill./Overall from Table C.15 "Overall" row
    "SAGE": {"Emp.": 4.12, "Flu.": 4.39, "Hum.": 4.28, "Info.": 4.05, "Exp.": 4.28, "Skill.": 4.08, "Overall": 4.12},
    "Qwen-2.5-7B": {"Emp.": 3.84, "Flu.": 4.28, "Hum.": 4.12, "Info.": 3.76, "Exp.": 3.91, "Skill.": 3.74, "Overall": 3.80},
    "Mistral-7B": {"Emp.": 3.62, "Flu.": 4.18, "Hum.": 3.95, "Info.": 3.71},
    "LLaMA-3.1-8B": {"Emp.": 3.54, "Flu.": 4.12, "Hum.": 3.87, "Info.": 3.65},
}
ESC_COLS = ["Flu.", "Exp.", "Emp.", "Info.", "Hum.", "Skill.", "Overall"]
GUESS = {"qwen": "Qwen-2.5-7B", "llama": "LLaMA-3.1-8B", "mistral": "Mistral-7B", "sage": "SAGE", "blender": "BlenderBot-Joint"}


def fmt(v):
    return "" if v is None else f"{v:.3f}" if abs(v) < 1 else f"{v:.2f}"

def main():
    rows = []
    for d in sorted(glob.glob("results/*/")):
        m = {}
        for name in ("metrics.json", "retrieval.json"):
            p = os.path.join(d, name)
            if os.path.exists(p):
                j = json.load(open(p))
                m.update(j.get("generation", {}), **j.get("strategy", {}), **j.get("retrieval", {}))
                m.update({k: v for k, v in j.items() if isinstance(v, (int, float))})
        run = os.path.basename(d.rstrip("/"))
        paper = None if run.lower().startswith("sage_") else next((v for k, v in GUESS.items() if k in run.lower()), None)
        rows.append((run, m, paper))
    print("| run | n | " + " | ".join(COLS) + " |")
    print("|---|---|" + "---|" * len(COLS))
    for run, m, paper in rows:
        print(f"| {run} (ours) | {m.get('n', '')} | " + " | ".join(fmt(m.get(c)) for c in COLS) + " |")
        if paper:
            print(f"| {paper} (paper) | | " + " | ".join(fmt(PAPER[paper].get(c)) for c in COLS) + " |")

    esc = sorted(glob.glob("results/esceval_*/esceval.json"))
    if esc:
        print("\nESC-Eval (ours: ESC-RANK/LLM judge, 0-4 shifted to 1-5; paper: human raters, 5-point)\n")
        print("| run | n | judge | " + " | ".join(ESC_COLS) + " |")
        print("|---|---|---|" + "---|" * len(ESC_COLS))
        for p in esc:
            j = json.load(open(p))
            run = os.path.basename(os.path.dirname(p))
            print(f"| {run} (ours) | {j['n']} | {j.get('judge', '')} | "
                  + " | ".join(fmt(j["sage_scale_1to5"].get(c)) for c in ESC_COLS) + " |")
            paper = next((v for k, v in GUESS.items() if k in run.lower().replace("esceval_", "")), None)
            if paper in ESC_PAPER:
                print(f"| {paper} (paper) | | human | " + " | ".join(fmt(ESC_PAPER[paper].get(c)) for c in ESC_COLS) + " |")

main()

## 9. Notes for the write-up

- These numbers come from a **re-implementation**. Read them against the assumptions in Sections 5-6 and the gaps listed in `AUDIT.md`.
- The retrieval scores depend on the candidate pool size, which the paper doesn't state. On the full test pool, BM25 gets R@10 ≈ 11, against 61.8 for the paper's DPR. We use a pool of 100.
- ESC-Eval here uses an automatic judge (ESC-RANK) on the 331 public English cards. SAGE reports three authors rating responses on a 5-point scale and claims 2,801 cards, so treat that comparison as indicative only.